#### API abfragen mit JSON und requests-Modul

- `requests` entspricht einem HTTP-Request
- Ein HTTP-Request (Hypertext Transfer Protocol Request) ist eine Nachricht, welche ein Client (z. B. ein Webbrowser) an einen Server sendet, um eine bestimmte Aktion auszuführen oder eine Ressource anzufordern. 

- die zentralen Eigenschaften und Bestandteile eines HTTP-Requests sind:
    1. HTTP-Version und Status code
        - Bsp.: `HTTP/1.1 200 OK` bedeutet:
            - HTTP Version: 1.1 
            - Status-Code 200: Erfolgscode. Die Ressource wurde erfolgreich übertragen. 
    2. HTTP-Methoden: 
        - __GET: Fordert eine Ressource vom Server an (sollte nur lesend sein).__

        - __POST: Sendet Daten an den Server (erstellt oft eine neue Ressource).__

        - PUT: Aktualisiert eine bestehende Ressource vollständig.

        - PATCH: Führt eine teilweise Aktualisierung einer Ressource durch.

        - DELETE: Löscht eine bestimmte Ressource.

        - __HEAD/OPTIONS: Dienen meist zur Metadaten-Abfrage oder zum Testen der Verbindung.__

        (PUT, PATCH, DELETE müssen vom Server erlaubt sein. Das ist nur bei speziellen Anwendungen der Fall)

    3. Request-URL: die Zieladresse, an die der Request gesendet wird
    4. HTTP-Header: Meta-Informationen z.B
        - Host: Der Domainname des Zielservers
        - Content-Type: Beschreibt das Format der im Body gesendeten Daten
        - User-Agent: Informationen über den Client (z. B. Browser-Typ)
        - u.v.m
    5. HTTP-Version
    6. Message Body
        - Die eigentlichen Daten die übertragen werden (HTML,JSON, XML, ..)





### `requests`-Modul installieren

In [ ]:
#1
#%pip install requests

### GET-Request zu JSON API (randomuser.me)

In [ ]:
#2
# GET-Request
import json
import requests


response = requests.get("https://randomuser.me/api/")
if response.status_code == 200:
    dict_random_user = json.loads(response.text)
else:
    print("Fehler beim laden der Resource")

# Hilfsfunktion
def jsonPrint(obj):
    print(json.dumps(obj,indent=4,sort_keys=True))


print(dict_random_user)
jsonPrint(dict_random_user)



#### JSON in CSV (strukturierte Daten) umwandeln

In [ ]:

#3

# import json
import requests

def jsonPrint(obj):
    print(json.dumps(obj,indent=4,sort_keys=True))

# Geographische Koordinaten von Berlin 
# Breitengrad (Latitude): 52.5186° N
# Längengrad (Longitude): 13.4083° E
lon = 52.5186
lat = 13.4083

response = requests.get(f"https://www.7timer.info/bin/astro.php?lon={lon:.3f}&lat={lon:.3f}&ac=0&unit=metric&output=json&tzshift=0")
if response.status_code == 200:
    data_obj = json.loads(response.text)
else:
    print("Fehler beim laden der Resource")


# Daten ausgeben
jsonPrint(data_obj)
#jsonPrint(data_obj['dataseries'])


# Interpretation
# "timepoint" beschreibt die Anzahl der Stunden, die seit dem Erstellungszeitpunkt des Datensatzes (init) vergangen sind.
#from datetime import datetime
#timestamp = int(data_obj['init'])
#dt_object = datetime.fromtimestamp(timestamp)
#print("Datum:", dt_object.strftime("%d.%m.%Y"))
#print("Uhrzeit:", dt_object.strftime("%H:%M:%S"))


In [ ]:
#4  
weather_data = data_obj
weather_data['dataseries'][0] # der Erste Datensatz

In [ ]:
#5 
#Spaltennamen für Headerzeile des CSV
weather_data['dataseries'][0].keys()

In [ ]:
#7
import csv

# 1. Keys aus dem ersten Datensatz extrahieren
keys = weather_data['dataseries'][0].keys()

# 2. Datei schreiben
# newline='' verhindert doppelte Zeilenumbrüche unter Windows
with open("test.csv", 'w', newline='', encoding='utf-8') as myfile:
    writer = csv.writer(myfile)
    
    # Header-Zeile schreiben
    writer.writerow(keys)
    
    # 3. Daten iterativ schreiben
    for entry in weather_data['dataseries']:
        row = [entry.get(k, '') for k in keys]
        writer.writerow(row)

#### Wir möchten die Spalte "wind10m" aufspalten in zwei Spalten `wind10m_direction` und `wind10m_speed`
- es ändern sich die mit #--> gekennzeichneten Zeilen

In [ ]:
#8

import csv

filename = 'weather.csv'
keys = list(weather_data['dataseries'][0].keys())

# Kopfzeile generieren 
keys_header = []
for k in keys:
    if k == 'wind10m':
        keys_header.extend(["wind10m_direction", "wind10m_speed"])
    else:
        keys_header.append(k)

# CSV Datei schreiben
with open(filename, "w", newline='', encoding='utf-8') as myfile:
    writer = csv.writer(myfile)
    writer.writerow(keys_header)
    
    # Datenzeilen verarbeiten
    for i in weather_data['dataseries']:
        row = []
        for j in keys:
            if j != 'wind10m':
                row.append(i.get(j, ''))
            else:
                # Hier greifen wir auf das geschachtelte Dictionary zu
                row.append(i[j].get('direction', ''))
                row.append(i[j].get('speed', ''))
        
        writer.writerow(row)